# per-rank-cuda-device — ex2: per-rank ctx dict with no-shared-device invariant

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `per-rank-cuda-device`. Running the final beacon cell reports progress against the `Distributed: per-rank cuda device` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: per-rank cuda device` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`per-rank-cuda-device`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "per-rank-cuda-device"
DD_SUBTOPIC = "Distributed: per-rank cuda device"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Per-rank context dict — device + stream + sync queue

Ex1 returned `(device, model, tensor)` for a single rank. Real training maintains a **per-rank context object** that wraps everything rank-specific so the rest of the training code is rank-oblivious:

```python
@dataclass
class RankContext:
    rank: int
    world_size: int
    device: torch.device
    stream: 'torch.cuda.Stream'   # one CUDA stream per rank
    is_master: bool               # rank == 0
```

**Why a dict/object, not 5 separate args.** Adding a 6th rank-specific field (e.g. a per-rank RNG generator) is a one-line change to the ctx definition vs. plumbing a new arg through every function in the training stack. ARENA-style flat args are pedagogically clean but scale badly past 3-4 fields.

**Invariant we drill here.** No two ranks share the same device index — `len({c.device.index for c in ctxs}) == world_size`. If this ever fails in a real run, two ranks are stomping on each other's CUDA context and OOM/training-collapse follows.

**`is_master` shortcut.** `ctx.is_master` reads better than `ctx.rank == 0` scattered across the codebase. Same invariant; the abstraction names it.

### Exercise 2 — per-rank ctx dict with no-shared-device invariant

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply per-rank `torch.device(f'cuda:{rank}')` construction inside a context-dict builder, then verify the no-shared-device invariant across all ranks under mocked CUDA.
> Keywords: per-rank-context, device, is_master, no-collision, mocked-cuda
> ```

**KCs targeted:** `build-rank-context-dict`, `no-two-ranks-share-device-index`

Implement `ex2_build_rank_ctx(rank, world_size)`. Build the per-rank training context as a dict:

1. Construct `device = t.device(f'cuda:{rank}')`. DO NOT use bare `'cuda'` or hardcode `'cuda:0'`.
2. Build the ctx dict with these keys:
   - `'rank'`: `rank`
   - `'world_size'`: `world_size`
   - `'device'`: the `torch.device` from step 1
   - `'is_master'`: `rank == 0` (bool)
   - `'device_str'`: `f'cuda:{rank}'` (the canonical string form)
3. Return the ctx dict.

This function does NOT call `torch.cuda.set_device` or `.to(...)` — it's pure context construction. The test runs it for every rank in `range(world_size)`, then asserts the no-collision invariant: `len({c['device'].index for c in ctxs}) == world_size`.

Input: `rank`, `world_size` — ints.
Output: `dict[str, Any]` with the 5 keys above.

In [ ]:
def ex2_build_rank_ctx(rank: int, world_size: int) -> dict:
    """Build per-rank context dict with rank, world_size, device, is_master, device_str."""
    raise NotImplementedError()


def _test_ex2():
    # Construct contexts for all 4 ranks (no CUDA needed — we never .to() anything).
    ctxs = [ex2_build_rank_ctx(rank=r, world_size=4) for r in range(4)]

    # Each ctx has the right keys.
    expected_keys = {'rank', 'world_size', 'device', 'is_master', 'device_str'}
    for r, ctx in enumerate(ctxs):
        assert isinstance(ctx, dict), f'rank {r}: expected dict, got {type(ctx).__name__}'
        assert set(ctx.keys()) == expected_keys, (
            f'rank {r}: keys {set(ctx.keys())} != expected {expected_keys}'
        )

    # rank + world_size threaded through.
    for r, ctx in enumerate(ctxs):
        assert ctx['rank'] == r, f'ctx["rank"] wrong on rank {r}'
        assert ctx['world_size'] == 4, f'ctx["world_size"] wrong on rank {r}'

    # device is a torch.device with the right index.
    for r, ctx in enumerate(ctxs):
        d = ctx['device']
        assert isinstance(d, t.device), f'rank {r}: device must be torch.device, got {type(d)}'
        assert d.type == 'cuda', f'rank {r}: expected cuda type, got {d.type!r}'
        assert d.index == r, f'rank {r}: expected cuda:{r}, got cuda:{d.index}'

    # device_str matches.
    for r, ctx in enumerate(ctxs):
        assert ctx['device_str'] == f'cuda:{r}', (
            f'rank {r}: device_str {ctx["device_str"]!r}, expected {f"cuda:{r}"!r}'
        )

    # is_master: True only on rank 0.
    assert ctxs[0]['is_master'] is True, 'rank 0 must have is_master=True'
    for r in [1, 2, 3]:
        assert ctxs[r]['is_master'] is False, f'rank {r} must have is_master=False'

    # *** THE LOAD-BEARING INVARIANT *** — no two ranks share device index.
    indices = {ctx['device'].index for ctx in ctxs}
    assert len(indices) == 4, (
        f'no-collision invariant FAILED — only {len(indices)} unique device indices '
        f'across 4 ranks. Indices: {sorted(c["device"].index for c in ctxs)}'
    )

    # Bigger world.
    ctxs8 = [ex2_build_rank_ctx(rank=r, world_size=8) for r in range(8)]
    indices8 = {ctx['device'].index for ctx in ctxs8}
    assert len(indices8) == 8, 'no-collision invariant FAILED at world_size=8'
    assert sum(1 for c in ctxs8 if c['is_master']) == 1, (
        f'exactly one master expected; got {sum(1 for c in ctxs8 if c["is_master"])}'
    )

    # Single-rank degenerate.
    ctx_solo = ex2_build_rank_ctx(rank=0, world_size=1)
    assert ctx_solo['device'].index == 0
    assert ctx_solo['is_master'] is True
    assert ctx_solo['world_size'] == 1

    # device_str + device must agree.
    for ctx in ctxs8:
        assert t.device(ctx['device_str']) == ctx['device'], (
            f'device_str {ctx["device_str"]!r} does not parse back to device {ctx["device"]!r}'
        )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_build_rank_ctx(rank: int, world_size: int) -> dict:
    device = t.device(f'cuda:{rank}')
    return {
        'rank': rank,
        'world_size': world_size,
        'device': device,
        'is_master': rank == 0,
        'device_str': f'cuda:{rank}',
    }
```

**Why the ctx dict, not 5 args.** Threading 5 arguments through every training-loop function call is painful and brittle. The ctx dict (or a frozen dataclass) is the canonical scaling form. Adding a 6th field (e.g. a per-rank RNG generator) becomes a one-line change.

**`torch.device` is constructible without CUDA being available.** Constructing `t.device('cuda:3')` on a CPU-only machine returns a `torch.device` object — only `.to(device)` or `torch.cuda.set_device(device)` actually touch the driver. This is what lets us run the test on Colab CPU.

**The no-collision invariant is the bug-catcher.** If a refactor accidentally pins all ranks to `cuda:0` (e.g. someone wrote `t.device('cuda')` instead of the f-string), all ranks land on the same GPU and OOM in the first batch. The set-size assertion in the test would catch this immediately.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()